# Boids: Swarming as Emergence (and Evolving It)
**DCS 340 · Project II starter notebook** · based on Reynolds (1987), *Flocks, Herds, and Schools: A Distributed Behavioral Model*

> ⚠️ Scaffold notebook. The `📝 Your notes` cells are for **you**: your predictions, observations, and connections to class. Those (and your blog/README) need to be in your own words per the GenAI policy.

**Map of the notebook**
1. The model: three local rules → one global behavior
2. Watching it run (animation)
3. Measuring emergence (polarization, spacing, group count)
4. 🧪 Class activity A: *Predict, then run* (rule knockouts)
5. 🧪 Class activity B: sliders / phase sweep
6. Bringing in Evolutionary Computing: a GA that evolves the rule weights
7. Bridge to Lehman & Stanley: objective vs. novelty search
8. 🧪 Extension challenges (predator, field of view, topological neighbors)

In [ ]:
# If running on Colab, everything below is preinstalled.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rcParams
from IPython.display import HTML, display
from scipy.sparse.csgraph import connected_components

rcParams['animation.embed_limit'] = 60  # MB, for longer animations
rng_global = np.random.default_rng(340)

## 1. The model

Each boid only sees neighbors inside a **perception radius** (and optionally a field of view). Reynolds' three behaviors, in his priority order:

| Reynolds' name (1987) | Common modern name | Steers toward… |
|---|---|---|
| Collision avoidance | **Separation** | away from boids that are too close |
| Velocity matching | **Alignment** | the average velocity of neighbors |
| Flock centering | **Cohesion** | the average position of neighbors |

**Steering, Reynolds-style:** each behavior produces a *desired velocity*; the steering force is `desired − current velocity`, capped at `max_force`.

**Two ways to combine the three forces** (a detail worth knowing for reproducibility):
- `mode="weighted"`: weighted sum, which is what most tutorials and libraries do.
- `mode="priority"`: Reynolds' *prioritized acceleration allocation*: fill an acceleration "budget" in priority order (separation first), so in a crowd, avoidance can crowd out the others.

📌 The world is a **torus** (wraps at the edges) to avoid wall effects.

In [ ]:
DEFAULTS = dict(
    n=80, size=100.0,
    radius=10.0,          # perception radius for alignment & cohesion
    sep_radius=4.0,       # "too close" radius for separation
    w_sep=1.5, w_ali=1.0, w_coh=1.0,
    max_speed=2.0, min_speed=0.5, max_force=0.08,
    fov_deg=360.0,        # field of view; 360 = sees all around
    mode="weighted",      # or "priority"
)

def _limit(v, max_mag):
    # Cap each row-vector's length at max_mag.
    mag = np.linalg.norm(v, axis=1, keepdims=True)
    scale = np.where(mag > max_mag, max_mag / np.maximum(mag, 1e-12), 1.0)
    return v * scale

def _steer_toward(desired_dir, vel, p, has):
    # Reynolds steering: desired velocity (at max speed) minus current velocity.
    mag = np.linalg.norm(desired_dir, axis=1, keepdims=True)
    desired = desired_dir / np.maximum(mag, 1e-12) * p["max_speed"]
    steer = _limit(desired - vel, p["max_force"])
    return np.where(has[:, None], steer, 0.0)

def init_flock(p, rng):
    pos = rng.uniform(0, p["size"], (p["n"], 2))
    ang = rng.uniform(0, 2*np.pi, p["n"])
    vel = np.c_[np.cos(ang), np.sin(ang)] * p["max_speed"] * 0.75
    return pos, vel

def pairwise(pos, size):
    # diff[i, j] = vector from boid i to boid j, using torus wrap-around.
    diff = pos[None, :, :] - pos[:, None, :]
    diff = (diff + size/2) % size - size/2
    dist = np.linalg.norm(diff, axis=2)
    return diff, dist

def step(pos, vel, p):
    diff, dist = pairwise(pos, p["size"])
    np.fill_diagonal(dist, np.inf)

    visible = dist < p["radius"]
    if p["fov_deg"] < 360:
        heading = vel / np.linalg.norm(vel, axis=1, keepdims=True)
        cosang = np.einsum('ik,ijk->ij', heading, diff) / np.maximum(dist, 1e-12)
        visible &= cosang > np.cos(np.radians(p["fov_deg"] / 2))
    too_close = visible & (dist < p["sep_radius"])

    cnt = visible.sum(1)
    has_n, has_close = cnt > 0, too_close.any(1)

    # Separation: push away, weighted by 1/distance (closer = stronger)
    away = -(diff / np.maximum(dist, 1e-12)[..., None]**2) * too_close[..., None]
    sep = _steer_toward(away.sum(1), vel, p, has_close)
    # Alignment: match average neighbor velocity
    avg_vel = (vel[None, :, :] * visible[..., None]).sum(1) / np.maximum(cnt, 1)[:, None]
    ali = _steer_toward(avg_vel, vel, p, has_n)
    # Cohesion: head toward neighbors' center (offset from me)
    center_off = (diff * visible[..., None]).sum(1) / np.maximum(cnt, 1)[:, None]
    coh = _steer_toward(center_off, vel, p, has_n)

    forces = [p["w_sep"]*sep, p["w_ali"]*ali, p["w_coh"]*coh]
    if p["mode"] == "weighted":
        acc = sum(forces)
    else:  # prioritized acceleration allocation (Reynolds 1987)
        acc = np.zeros_like(vel)
        budget = np.full(len(vel), p["max_force"] * 1.5)
        for f in forces:  # priority order: separation, alignment, cohesion
            fm = np.linalg.norm(f, axis=1)
            take = np.minimum(fm, budget)
            acc += f * (take / np.maximum(fm, 1e-12))[:, None]
            budget -= take

    vel = vel + acc
    speed = np.linalg.norm(vel, axis=1, keepdims=True)
    vel = vel / np.maximum(speed, 1e-12) * np.clip(speed, p["min_speed"], p["max_speed"])
    pos = (pos + vel) % p["size"]
    return pos, vel

def simulate(steps=300, seed=0, record=True, **overrides):
    p = {**DEFAULTS, **overrides}
    rng = np.random.default_rng(seed)
    pos, vel = init_flock(p, rng)
    hist = []
    for _ in range(steps):
        pos, vel = step(pos, vel, p)
        if record:
            hist.append((pos.copy(), vel.copy()))
    return (hist if record else (pos, vel)), p

## 2. Watching it run

In [ ]:
def animate(hist, p, every=2, title=""):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(0, p["size"]); ax.set_ylim(0, p["size"])
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')
    ax.set_title(title)
    pos, vel = hist[0]
    q = ax.quiver(pos[:, 0], pos[:, 1], vel[:, 0], vel[:, 1],
                  angles='xy', scale_units='xy', scale=0.4, width=0.006, color='tab:blue')
    def update(k):
        pos, vel = hist[k * every]
        q.set_offsets(pos); q.set_UVC(vel[:, 0], vel[:, 1])
        return (q,)
    anim = animation.FuncAnimation(fig, update, frames=len(hist)//every, interval=40, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

hist, p = simulate(steps=300, seed=1)
animate(hist, p, title="Default boids")

📝 **Your notes:** What did you expect before you ran it? What surprised you? Where is the "leader"? (There isn't one. How would you explain that to someone in one sentence?)

## 3. Measuring emergence

"It looks like a flock" isn't a measurement. Three numbers that make the emergence quantitative:

- **Polarization** $\phi = \left\|\frac{1}{N}\sum_i \hat v_i\right\|$: 0 = headings random, 1 = everyone moving the same direction. (This is the order parameter from Vicsek et al. 1995.)
- **Mean nearest-neighbor distance (NND):** how tightly packed.
- **Number of groups:** connected components when boids within a *fixed* link distance (6 units) are linked. (Fixed on purpose: an earlier version used each flock's own perception radius, and the GA below "cheated" by maxing out the radius so everything counted as one group.)

In [ ]:
LINK_DIST = 6.0

def metrics(pos, vel, p):
    heading = vel / np.linalg.norm(vel, axis=1, keepdims=True)
    polar = np.linalg.norm(heading.mean(0))
    _, dist = pairwise(pos, p["size"])
    np.fill_diagonal(dist, np.inf)
    nnd = dist.min(1).mean()
    n_groups, labels = connected_components(dist < LINK_DIST, directed=False)
    largest = np.bincount(labels).max() / len(pos)
    collisions = (dist < 1.0).sum() / 2      # pairs closer than 1 unit
    return dict(polarization=polar, nnd=nnd, groups=n_groups,
                largest_frac=largest, collisions=collisions)

def metric_traces(hist, p):
    m = [metrics(pos, vel, p) for pos, vel in hist]
    return {k: np.array([d[k] for d in m]) for k in m[0]}

tr = metric_traces(hist, p)
fig, axs = plt.subplots(1, 3, figsize=(13, 3))
for ax, k in zip(axs, ["polarization", "nnd", "groups"]):
    ax.plot(tr[k]); ax.set_title(k); ax.set_xlabel("time step")
plt.tight_layout(); plt.show()

## 4. 🧪 Class activity A: *Predict, then run*

**Instructions for the room:** for each mystery flock, write down a prediction *before* running: Will it align? Clump? Scatter? How many groups?

Then run the cell and compare against the metrics.

In [ ]:
experiments = {
    "A. Separation only":        dict(w_sep=1.5, w_ali=0.0, w_coh=0.0),
    "B. Alignment only":         dict(w_sep=0.0, w_ali=1.0, w_coh=0.0),
    "C. Cohesion only":          dict(w_sep=0.0, w_ali=0.0, w_coh=1.0),
    "D. No separation":          dict(w_sep=0.0, w_ali=1.0, w_coh=1.0),
    "E. No alignment":           dict(w_sep=1.5, w_ali=0.0, w_coh=1.0),
    "F. All three (baseline)":   dict(),
}

fig, axs = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, ov) in zip(axs.flat, experiments.items()):
    (pos, vel), p = simulate(steps=400, seed=2, record=False, **ov)
    m = metrics(pos, vel, p)
    ax.quiver(pos[:, 0], pos[:, 1], vel[:, 0], vel[:, 1], angles='xy',
              scale_units='xy', scale=0.4, width=0.006)
    ax.set_xlim(0, p["size"]); ax.set_ylim(0, p["size"]); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name}\nφ={m['polarization']:.2f}  groups={m['groups']}  NND={m['nnd']:.1f}", fontsize=9)
plt.tight_layout(); plt.show()

# Want to watch one? e.g.:
# h, p = simulate(steps=300, seed=2, **experiments["D. No separation"]); animate(h, p)

📝 **Your notes / discussion prompts**
- Which single rule is *necessary* for global alignment? Which one prevents collapse?
- Alignment-only still produces order. What does that suggest about where "flocking" really comes from?
- Is "no separation" realistic for real birds? Why did Reynolds put collision avoidance *first* in priority?

## 5. 🧪 Class activity B: sliders and a phase sweep

Run the slider cell, then drag. **Challenge for groups:** find settings that produce (a) one tight ball, (b) several separate flocks, (c) a "river" (high polarization, spread out), (d) total chaos.

In [ ]:
try:
    import ipywidgets as w
    def explore(w_sep=1.5, w_ali=1.0, w_coh=1.0, radius=10.0, fov_deg=360, mode="weighted"):
        (pos, vel), p = simulate(steps=300, seed=3, record=False, w_sep=w_sep, w_ali=w_ali,
                                 w_coh=w_coh, radius=radius, fov_deg=fov_deg, mode=mode)
        m = metrics(pos, vel, p)
        plt.figure(figsize=(4.5, 4.5))
        plt.quiver(pos[:, 0], pos[:, 1], vel[:, 0], vel[:, 1], angles='xy', scale_units='xy', scale=0.4)
        plt.xlim(0, p["size"]); plt.ylim(0, p["size"]); plt.gca().set_aspect('equal')
        plt.title(" ".join(f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}" for k, v in m.items()), fontsize=8)
        plt.show()
    w.interact_manual(explore,
        w_sep=(0.0, 4.0, 0.25), w_ali=(0.0, 4.0, 0.25), w_coh=(0.0, 4.0, 0.25),
        radius=(2.0, 30.0, 1.0), fov_deg=(60, 360, 30), mode=["weighted", "priority"])
except ImportError:
    print("ipywidgets not available: edit the parameters in explore(...) directly.")

**Phase sweep:** how does global order depend on perception radius? (Local information → global structure.)

In [ ]:
radii = [2, 4, 6, 8, 10, 14, 18, 25]
seeds = range(3)
phi = [[metric_traces(simulate(steps=250, seed=s, radius=r)[0][-50:], {**DEFAULTS, "radius": r})["polarization"].mean()
        for s in seeds] for r in radii]
phi = np.array(phi)
plt.errorbar(radii, phi.mean(1), yerr=phi.std(1), marker='o', capsize=3)
plt.xlabel("perception radius"); plt.ylabel("polarization (last 50 steps)")
plt.title("How much do boids need to see to flock?"); plt.show()

📝 **Your notes:** Is there a threshold? What's the analogy to a phase transition (cf. Vicsek model)?

## 6. Bringing in Evolutionary Computing

Reynolds hand-tuned his parameters. Can a **genetic algorithm** find them instead?

- **Genome:** `[w_sep, w_ali, w_coh, radius]`
- **Fitness (one possible objective; question it!):** high polarization + most boids in one group − collisions, averaged over the whole run (so flocks that form *quickly* score higher)
- **Operators:** tournament selection, blend crossover, Gaussian mutation, elitism (same pieces as our earlier class GAs)

In [ ]:
BOUNDS = np.array([[0, 4], [0, 4], [0, 4], [2, 25]])   # w_sep, w_ali, w_coh, radius
GENE_NAMES = ["w_sep", "w_ali", "w_coh", "radius"]

def run_genome(g, seed, steps=150, n=50):
    hist, p = simulate(steps=steps, seed=seed, n=n,
                       w_sep=g[0], w_ali=g[1], w_coh=g[2], radius=g[3])
    tr = metric_traces(hist[::5], p)   # average over the WHOLE run, so flocking *fast* is rewarded
    return {k: v.mean() for k, v in tr.items()}

def objective(m):
    return m["polarization"] + m["largest_frac"] - 0.05 * m["collisions"]

def behavior(m):
    # Behavior descriptor used later for novelty search
    return np.array([m["polarization"], m["largest_frac"], m["nnd"] / 10])

def ga(fitness_fn, pop_size=16, gens=12, seed=0, mut_sd=0.15, elite=2):
    rng = np.random.default_rng(seed)
    pop = rng.uniform(BOUNDS[:, 0], BOUNDS[:, 1], (pop_size, 4))
    log = []
    for gen in range(gens):
        ms = [run_genome(g, seed=gen) for g in pop]
        fit = np.array(fitness_fn(ms, pop))
        order = np.argsort(-fit)
        log.append(dict(gen=gen, best=fit[order[0]], mean=fit.mean(),
                        best_genome=pop[order[0]].copy(), metrics=ms, pop=pop.copy()))
        print(f"gen {gen:2d}  best={fit[order[0]]:.3f}  mean={fit.mean():.3f}  "
              f"best genome={np.round(pop[order[0]], 2)}")
        new = [pop[i].copy() for i in order[:elite]]
        while len(new) < pop_size:
            a, b = [max(rng.choice(pop_size, 3), key=lambda i: fit[i]) for _ in range(2)]
            alpha = rng.uniform(-0.25, 1.25, 4)                 # blend crossover
            child = pop[a] + alpha * (pop[b] - pop[a])
            child += rng.normal(0, mut_sd, 4) * (BOUNDS[:, 1] - BOUNDS[:, 0])
            new.append(np.clip(child, BOUNDS[:, 0], BOUNDS[:, 1]))
        pop = np.array(new)
    return log

obj_log = ga(lambda ms, pop: [objective(m) for m in ms])

In [ ]:
plt.plot([l["best"] for l in obj_log], label="best")
plt.plot([l["mean"] for l in obj_log], label="mean")
plt.xlabel("generation"); plt.ylabel("fitness"); plt.legend(); plt.show()

best = obj_log[-1]["best_genome"]
print(dict(zip(GENE_NAMES, np.round(best, 2))))
h, p = simulate(steps=300, seed=10, w_sep=best[0], w_ali=best[1], w_coh=best[2], radius=best[3])
animate(h, p, title="GA-evolved flock")

📝 **Your notes / discussion prompts**
- Does the evolved flock *look* like a flock, or did the GA find a loophole in the fitness function? (Look closely at the genome. Is any weight near 0 or the max?)
- What did the fitness function **reward** vs. what you **meant**? This is the core of the Lehman & Stanley reading.
- Try changing `objective`, e.g. reward *multiple* groups, or penalize clumping (low NND).

## 7. Bridge to Lehman & Stanley: novelty search

Instead of rewarding an objective, reward being **different** from what has been seen before. Each genome's *behavior* = (polarization, largest-group fraction, spacing). Novelty = mean distance to its *k* nearest neighbors among the current population + an archive.

In [ ]:
archive = []
def novelty_fitness(ms, pop, k=5):
    B = np.array([behavior(m) for m in ms])
    ref = np.vstack([B] + ([np.array(archive)] if archive else []))
    d = np.linalg.norm(B[:, None] - ref[None], axis=2)
    nov = np.sort(d, axis=1)[:, 1:k+1].mean(1)          # skip self (distance 0)
    for i in np.argsort(-nov)[:2]:                       # add the 2 most novel to archive
        archive.append(B[i])
    return nov

nov_log = ga(novelty_fitness)

In [ ]:
def all_behaviors(log):
    return np.array([behavior(m) for l in log for m in l["metrics"]])

Bo, Bn = all_behaviors(obj_log), all_behaviors(nov_log)
plt.figure(figsize=(6, 5))
plt.scatter(Bo[:, 0], Bo[:, 2], s=12, alpha=.5, label="objective GA")
plt.scatter(Bn[:, 0], Bn[:, 2], s=12, alpha=.5, label="novelty search")
plt.xlabel("polarization"); plt.ylabel("spacing (NND/10)")
plt.title("Which search explored more kinds of collective behavior?"); plt.legend(); plt.show()

📝 **Your notes:** Where did each search spend its time? Which would you use to *discover* collective behaviors nobody designed, vs. to *engineer* one you already want (e.g. a drone swarm)?

## 8. 🧪 Extension challenges (pick one in small groups)

1. **Predator:** add a point that moves through the world; boids within some radius get a strong flee force (highest priority). Do you see the flock split and re-merge ("fountain effect")?
2. **Field of view:** set `fov_deg=270` or `120`. Real birds have blind spots. How does it change polarization?
3. **Topological neighbors:** Ballerini et al. (2008) found starlings track a fixed *number* of neighbors (~6–7), not everyone within a distance. Modify `step` so `visible` = each boid's 7 nearest. What changes when density changes?
4. **Evolve something new:** add `fov_deg` or `max_force` to the genome.

In [ ]:
# Challenge 1 starter: predator.
# Hint: compute the torus-wrapped vector from each boid to the predator,
# make a flee steering force with _steer_toward(-to_pred, vel, p, near_mask),
# and add it with a big weight (or put it first in the priority list).

def step_with_predator(pos, vel, p, pred_pos, flee_radius=15.0, w_flee=3.0):
    pos, vel = step(pos, vel, p)        # normal boids update
    # TODO: your code here
    return pos, vel

In [ ]:
# Challenge 3 starter: topological neighbors.
# Hint: in step(), replace `visible = dist < p["radius"]` with a mask that is True
# for each row's k smallest distances, e.g. using np.argsort(dist, axis=1)[:, :k].

## References & further reading (verify before citing!)
- Reynolds, C. W. (1987). Flocks, herds and schools: A distributed behavioral model. *SIGGRAPH '87*. https://dl.acm.org/doi/10.1145/37402.37406
- Reynolds' Boids page: https://www.red3d.com/cwr/boids/
- Reynolds, C. W. (1999). Steering behaviors for autonomous characters. *GDC*.
- Vicsek, T. et al. (1995). Novel type of phase transition in a system of self-driven particles. *Physical Review Letters*.
- Couzin, I. D. et al. (2002). Collective memory and spatial sorting in animal groups. *Journal of Theoretical Biology*.
- Ballerini, M. et al. (2008). Interaction ruling animal collective behavior depends on topological rather than metric distance. *PNAS*.
- Kennedy, J. & Eberhart, R. (1995). Particle swarm optimization. *IEEE ICNN*.
- Lehman, J. & Stanley, K. O. (2011). Abandoning objectives: Evolution through the search for novelty alone. *Evolutionary Computation*.